# 59. `PreparedAmplitudeCache` used directly

**Objectives:**

- Build a `PreparedAmplitudeCache` directly with `DecayModel.prepare_cache(...)`.
- Call `.evaluate()`, `.amplitude()`, `.intensity()`, `.normalization()` and
  `.normalization_matrix()` on it and contrast them with `DecayModel.amplitude`/
  `.intensity`, which only cover the unnormalized data-side amplitude.
- Read the fixed/floating DYNAMICS split from the class's own docstring and see
  `check_parameters` catch a stale parameter list.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag,
    Resonance, generate_toy,
)

## 1. A small model with one floating dynamics parameter

The rho mass is a `Parameter.dynamics(..., owner="rho")`; everything else is fixed.
`PreparedAmplitudeCache.prepare()` (called for us by `DecayModel.prepare_cache`) bakes,
once, which components are affected by a floating `ParameterKind.DYNAMICS` parameter --
here only `"rho"` -- and folds every other component into a fixed
`data_components`/`normalization_matrix_fixed` block that later evaluations never
recompute.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
mass = Parameter.dynamics("rho.mass", 0.7753, owner="rho", bounds=(0.73, 0.82), step=0.001)
model = DecayModel(
    channel,
    [Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=mass, width=0.1491, spin=1),
     NonResonant(RealImag(0.55, 0.30), name="NR")],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=60,
)
truth = {"rho.mass": 0.7753}
data = generate_toy(
    model, 1200, parameters=truth, seed=59,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
print(f"Generated {data.size} events")

Generated 1200 events


## 2. Build the cache and inspect the fixed/floating split

`prepare_cache(data_sample, normalization_sample)` is `DecayModel`'s public entry
point onto `PreparedAmplitudeCache.prepare()`. `floating_dynamic_owners` reports the
component names the cache will recompute when their DYNAMICS parameters change;
everything else stays in the fixed block for the lifetime of this cache.

In [3]:
cache = model.prepare_cache(data, model.normalization_sample)
print("Components:", [c.name for c in cache.components])
print("Floating dynamic owners:", cache.floating_dynamic_owners)
print("is_compact (no floating dynamics at all):", cache.is_compact)

Components: ['rho', 'NR']
Floating dynamic owners: frozenset({'rho'})
is_compact (no floating dynamics at all): False


## 3. `.evaluate()` / `.amplitude()` / `.intensity()` vs `DecayModel.amplitude`/`.intensity`

`DecayModel.amplitude`/`.intensity` recompute every component's dynamics from scratch
on whatever `data` mapping they are given -- there is no cache. `cache.amplitude`/
`.intensity` reuse the fixed block and only re-evaluate `"rho"` from `fit_values`,
but the *values* returned on the same data sample are identical. `.evaluate()`
returns `(intensity, normalization)` together in one call, which is the pair a
likelihood needs.

In [4]:
fit_values = {"rho.mass": 0.7753}

amplitude_cache = cache.amplitude(fit_values)
amplitude_model = model.amplitude(data.as_dict(), fit_values)
np.testing.assert_allclose(amplitude_cache, amplitude_model, atol=1e-10)

intensity_cache = cache.intensity(fit_values)
intensity_model = model.intensity(data.as_dict(), fit_values)
np.testing.assert_allclose(intensity_cache, intensity_model, atol=1e-10)

intensity_evaluate, normalization_evaluate = cache.evaluate(fit_values)
np.testing.assert_allclose(intensity_evaluate, intensity_cache, atol=1e-10)
print("cache.amplitude/.intensity match DecayModel.amplitude/.intensity on `data`.")
print(f"normalization(fit_values) = {float(normalization_evaluate):.6f}")

cache.amplitude/.intensity match DecayModel.amplitude/.intensity on `data`.
normalization(fit_values) = 1.392500


## 4. `.normalization()` and `.normalization_matrix()`

`DecayModel` has no public bare `normalization(...)` method: the total normalization
integral is otherwise only available implicitly, e.g. inside `SignalPDF`/`FitSession`.
The cache exposes it directly as `c^dagger M c`, with `M` the Hermitian
`normalization_matrix()` (`M_ij = integral conj(F_i) F_j dPhi`, CLAUDE.md's
"Normalization" section). We reproduce `cache.normalization` by hand from the same
`normalization_sample` `DecayModel` already uses, confirming this is exactly the
integral `FitSession` would divide the intensity by.

In [5]:
matrix = cache.normalization_matrix(fit_values)
print("normalization_matrix shape:", matrix.shape)
print("Hermitian:", bool(jnp.allclose(matrix, jnp.conj(matrix).T, atol=1e-10)))
print("Positive diagonal:", bool(jnp.all(jnp.real(jnp.diag(matrix)) > 0)))

norm_sample = model.normalization_sample
manual_normalization = jnp.mean(
    norm_sample.weights * model.intensity(norm_sample.as_dict(), fit_values)
)
np.testing.assert_allclose(cache.normalization(fit_values), manual_normalization, rtol=1e-8)
print("cache.normalization(fit_values) matches mean(weights * DecayModel.intensity(...)).")

normalization_matrix shape: (2, 2)
Hermitian: True


Positive diagonal: True


cache.normalization(fit_values) matches mean(weights * DecayModel.intensity(...)).


## 5. `check_parameters` catches a stale parameter list

The cache's own docstring warns: if a caller assembles `Minimizer` with a *different*
`Parameter` list that disagrees on which DYNAMICS parameters are floating, evaluation
silently keeps ignoring that parameter -- a structural zero gradient, not an error.
`check_parameters` is the guard against this. Fixing `rho.mass` in a copy of the
parameter list (while the cache was prepared with it floating) must raise.

In [6]:
import dataclasses

cache.check_parameters(model.parameters)  # the same list used to prepare(): passes silently
print("check_parameters passed for the parameter list the cache was prepared with.")

stale = tuple(
    dataclasses.replace(p, fixed=True) if p.name == "rho.mass" else p
    for p in model.parameters
)
try:
    cache.check_parameters(stale)
    raise AssertionError("expected check_parameters to reject the stale parameter list")
except ValueError as exc:
    print("check_parameters correctly rejected a stale list:")
    print(exc)

check_parameters passed for the parameter list the cache was prepared with.
check_parameters correctly rejected a stale list:
parameters are inconsistent with the dynamics this cache was prepared with: components [] are floating in `parameters` but were fixed when this cache was prepared; components ['rho'] are fixed in `parameters` but were floating at prepare time. Rebuild the cache with PreparedAmplitudeCache.prepare(..., parameters=parameters) or pass the same parameter list used to prepare the cache to Minimizer.


## Continue learning

`PreparedAmplitudeCache` is the object `FitSession`/`DecayModel.pdf()` build for you
internally; see CLAUDE.md's "Prepared caching is a performance-critical pattern"
section and [`docs/performance.md`](../../docs/performance.md) for what is cached vs.
recomputed on each Minuit step, and
[tutorial 5](tutorial_05_dynamics_and_low_level_api.ipynb) for the same cache driving
an explicit `MultiBackgroundNLL`/`Minimizer` fit.

Return to [the course guide](TUTORIALS.md).